# 🤟 ISL Gesture Recognition — Definitive Training Pipeline (No-Error Version)

**Storage-optimised design:**
- 💾 Processed landmarks (~400MB) → saved to **Google Drive** permanently.
- 📥 Raw videos (57GB) → downloaded only as needed and **automatically deleted** from temp space.
- 🔁 **Resume Support:** If Colab disconnects, it will automatically skip everything it already did!

---
### ⚠️ CRITICAL SETUP BEFORE RUNNING
1. Go to `Runtime → Change runtime type → T4 GPU`.
2. Ensure you have ~500MB free space on your Google Drive.
3. **Run Cell 1 (Drive Mount)** and follow the prompt to allow access.

In [ ]:
# ─── Cell 1: Mount Google Drive ───────────────────────────────────────────────
import os
from google.colab import drive
drive.mount('/content/drive')
print('\n✅ Google Drive mounted successfully!')

In [ ]:
# ─── Cell 2: Install Libraries (Robust Version) ───────────────────────────────
!pip install -q mediapipe opencv-python-headless tensorflow==2.15.0 scikit-learn numpy

import mediapipe as mp
import tensorflow as tf
import numpy as np
import cv2
print(f'✅ MediaPipe version: {mp.__version__}')
print(f'✅ TensorFlow version: {tf.__version__}')

# Pre-check Holistic API
try:
    _ = mp.solutions.holistic
    print('✅ MediaPipe Holistic is ready for use!')
except AttributeError:
    print('❌ Error: MediaPipe Holistic not found. Restart runtime and try again.')

In [ ]:
# ─── Cell 3: Configuration & Folder Setup ─────────────────────────────────────
import shutil, sys, time, urllib.request, zipfile, glob

# ─ Paths ─────────────────────────────────────────────────────────────────────
DRIVE_ROOT = '/content/drive/MyDrive/ISL_Project'
DATA_DIR   = f'{DRIVE_ROOT}/include_data'    # Landmark files go here (~400MB total)
VIDEO_DIR  = '/content/include_videos'       # Videos go here (temp, will be deleted)

for d in [DRIVE_ROOT, DATA_DIR, VIDEO_DIR]:
    os.makedirs(d, exist_ok=True)

# ─ Features ──────────────────────────────────────────────────────────────────
NUM_HAND_LANDMARKS    = 21
NUM_POSE_LANDMARKS    = 12
POSE_LANDMARK_INDICES = [11, 12, 13, 14, 15, 16, 23, 24, 25, 26, 27, 28]
SEQUENCE_LENGTH       = 30
STEP_SIZE             = 10
NUM_FEATURES          = (NUM_HAND_LANDMARKS * 3 * 2) + (NUM_POSE_LANDMARKS * 3) # 162

print(f'✅ Configuration ready. Landmarks will be saved to: {DATA_DIR}')

In [ ]:
# ─── Cell 4: Download & Process One-By-One (Resumable) ──────────────────────────
ZENODO_BASE = 'https://zenodo.org/records/4010759/files'
DATASET_CATEGORIES = {
    'Adjectives':              ['Adjectives_1of8.zip','Adjectives_2of8.zip','Adjectives_3of8.zip',
                                'Adjectives_4of8.zip','Adjectives_5of8.zip','Adjectives_6of8.zip',
                                'Adjectives_7of8.zip','Adjectives_8of8.zip'],
    'Animals':                 ['Animals_1of2.zip','Animals_2of2.zip'],
    'Clothes':                 ['Clothes_1of2.zip','Clothes_2of2.zip'],
    'Colours':                 ['Colours_1of2.zip','Colours_2of2.zip'],
    'Days_and_Time':           ['Days_and_Time_1of3.zip','Days_and_Time_2of3.zip','Days_and_Time_3of3.zip'],
    'Electronics':             ['Electronics_1of2.zip','Electronics_2of2.zip'],
    'Greetings':               ['Greetings_1of2.zip','Greetings_2of2.zip'],
    'Home':                    ['Home_1of4.zip','Home_2of4.zip','Home_3of4.zip','Home_4of4.zip'],
    'Jobs':                    ['Jobs_1of2.zip','Jobs_2of2.zip'],
    'Means_of_Transportation': ['Means_of_Transportation_1of2.zip','Means_of_Transportation_2of2.zip'],
    'People':                  ['People_1of5.zip','People_2of5.zip','People_3of5.zip',
                                'People_4of5.zip','People_5of5.zip'],
    'Places':                  ['Places_1of4.zip','Places_2of4.zip','Places_3of4.zip','Places_4of4.zip'],
    'Pronouns':                ['Pronouns_1of2.zip','Pronouns_2of2.zip'],
    'Seasons':                 ['Seasons_1of1.zip'],
    'Society':                 ['Society_1of3.zip','Society_2of3.zip','Society_3of3.zip'],
}

def normalize_to_body(lh, rh, pose, pose_lms):
    if pose_lms is None or not (11 < len(pose_lms.landmark) and 12 < len(pose_lms.landmark)):
        return lh, rh, pose
    ls, rs = pose_lms.landmark[11], pose_lms.landmark[12]
    ref = np.array([(ls.x+rs.x)/2, (ls.y+rs.y)/2, (ls.z+rs.z)/2], dtype=np.float32)
    dist = max(float(np.sqrt((ls.x-rs.x)**2 + (ls.y-rs.y)**2 + (ls.z-rs.z)**2)), 1e-6)
    def norm(arr, n):
        if np.all(arr == 0): return arr
        return ((arr.reshape(n, 3) - ref) / dist).flatten().astype(np.float32)
    return norm(lh, 21), norm(rh, 21), norm(pose, 12)

def extract_from_video(v_path, holistic):
    cap = cv2.VideoCapture(v_path); frames = []
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret: break
        rgb = cv2.cvtColor(cv2.resize(frame, (640, 480)), cv2.COLOR_BGR2RGB)
        res = holistic.process(rgb)
        lh = np.array([[l.x,l.y,l.z] for l in res.left_hand_landmarks.landmark]).flatten() if res.left_hand_landmarks else np.zeros(63)
        rh = np.array([[l.x,l.y,l.z] for l in res.right_hand_landmarks.landmark]).flatten() if res.right_hand_landmarks else np.zeros(63)
        ps = np.array([[res.pose_landmarks.landmark[i].x, res.pose_landmarks.landmark[i].y, res.pose_landmarks.landmark[i].z] for i in POSE_LANDMARK_INDICES]).flatten() if res.pose_landmarks else np.zeros(36)
        lh, rh, ps = normalize_to_body(lh, rh, ps, res.pose_landmarks)
        frames.append(np.concatenate([lh, rh, ps]))
    cap.release()
    if len(frames) < 5: return []
    if len(frames) <= 30:
        s = np.zeros((30, 162), dtype=np.float32); s[:len(frames)] = frames; return [s]
    return [np.array(frames[s:s+30], dtype=np.float32) for s in range(0, len(frames)-30+1, 10)]

with mp.solutions.holistic.Holistic() as holistic:
    for cat_idx, (cat_name, zips) in enumerate(DATASET_CATEGORIES.items()):
        print(f'\nProcessing [{cat_idx+1}/15]: {cat_name}')
        cat_dir = os.path.join(VIDEO_DIR, cat_name)
        
        # Check if entire category is already processed
        cat_done = True
        for zname in zips: # Simple check: at least some folders exist
            if not os.path.isdir(os.path.join(DATA_DIR, cat_name.upper())): cat_done = False
        
        for zname in zips:
            z_p = f'/content/{zname}'
            if not os.path.exists(z_p):
                print(f'  Downloading {zname}...')
                urllib.request.urlretrieve(f'{ZENODO_BASE}/{zname}?download=1', z_p)
            with zipfile.ZipFile(z_p, 'r') as zf: zf.extractall(VIDEO_DIR)
            os.remove(z_p)

        for w_n in sorted(os.listdir(cat_dir)):
            w_p = os.path.join(cat_dir, w_n); w_k = w_n.strip().upper().replace(' ', '_')
            vids = glob.glob(os.path.join(w_p, '*.mp4')) + glob.glob(os.path.join(w_p, '*.avi'))
            if not vids: continue
            o_d = os.path.join(DATA_DIR, w_k); os.makedirs(o_d, exist_ok=True)
            if len(os.listdir(o_d)) >= len(vids): continue # Skip cached
            
            for i, v_p in enumerate(vids):
                for j, s in enumerate(extract_from_video(v_p, holistic)):
                    np.save(os.path.join(o_d, f's_{i}_{j}.npy'), s)
            print(f'    ✓ {w_k} done')
        shutil.rmtree(cat_dir)
print('\n✅ All extraction complete!')

In [ ]:
# ─── Cell 5: Train Model ──────────────────────────────────────────────────────
from sklearn.model_selection import train_test_split
from tensorflow import keras

X, y, words = [], [], sorted([d for d in os.listdir(DATA_DIR) if os.path.isdir(os.path.join(DATA_DIR, d))])
w2i = {w: i for i, w in enumerate(words)}

print('Loading samples from Drive...')
for w in words:
    w_d = os.path.join(DATA_DIR, w)
    for f in os.listdir(w_d):
        if f.endswith('.npy'):
            X.append(np.load(os.path.join(w_d, f))); y.append(w2i[w])

X, y = np.array(X), np.array(y)
y_c = keras.utils.to_categorical(y, len(words))
X_t, X_v, y_t, y_v = train_test_split(X, y_c, test_size=0.15, stratify=y)

model = keras.Sequential([
    keras.layers.Input(shape=(30, 162)),
    keras.layers.LSTM(128, return_sequences=True),
    keras.layers.LSTM(128, return_sequences=True),
    keras.layers.LSTM(64),
    keras.layers.Dense(len(words), activation='softmax')
])
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
model.fit(X_t, y_t, validation_data=(X_v, y_v), epochs=100, batch_size=64, verbose=1)
model.save('/content/isl_model.keras')
with open('/content/words.json', 'w') as f: json.dump({'words': words}, f)
print('\n✅ Training finished! Model saved to /content/isl_model.keras')

In [ ]:
# ─── Cell 6: Download Files to Mac ───────────────────────────────────────────
from google.colab import files
files.download('/content/isl_model.keras')
files.download('/content/words.json')
print('📦 Downloading results...')